# Credora ML-01: Baseline Logistic Regression for Loan Pre-Screening

This interactive notebook demonstrates the complete, end-to-end Machine Learning baseline pipeline for Credora's automated loan pre-screening system, implemented according to the architectural design in `Logistic Regression Model _Credora.drawio`.

### Pipeline Steps:
1. **Problem Formulation & Business Context**
2. **Data Ingestion & Exploratory Data Analysis (EDA)**
3. **Data Cleaning & Missing Value Imputation**
4. **Feature Engineering (One-Hot Encoding & Dependent Preference Weight)**
5. **Stratified Train-Test Splitting & Feature Scaling**
6. **Model Training (L2 Regularized Logistic Regression)**
7. **Credit Risk Evaluation (Confusion Matrix, Precision, Recall, F1, ROC-AUC)**
8. **Model Interpretability (Coefficients & Odds Ratios)**
9. **Artifact Serialization for Production Serving**

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries loaded successfully!")

## 1. Load Raw Dataset

In [ ]:
data_path = "../data/raw_loan_data.csv"
df = pd.read_csv(data_path)
print(f"Raw dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis & Target Distribution

In [ ]:
print("Target Variable Distribution:")
print(df['Loan_Approved'].value_counts(dropna=False))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Loan_Approved', palette=['#E11D48', '#0066FF'])
plt.title("Loan Approval Distribution (Class Imbalance Analysis)")
plt.xlabel("Loan Approved")
plt.ylabel("Count")
plt.show()

## 3. Data Cleaning & Missing Value Imputation
- Drop identifier `Applicant_ID`
- Drop rows with missing target values
- Impute missing numeric values using column medians
- Impute missing categorical values using column modes

In [ ]:
# Drop ID
if 'Applicant_ID' in df.columns:
    df = df.drop(columns=['Applicant_ID'])

# Drop unlabelled target rows
df = df.dropna(subset=['Loan_Approved'])

numeric_cols = [c for c in df.columns if c != 'Loan_Approved' and pd.api.types.is_numeric_dtype(df[c])]
categorical_cols = [c for c in df.columns if c != 'Loan_Approved' and not pd.api.types.is_numeric_dtype(df[c])]

impute_vals = {'numeric': {}, 'categorical': {}}
for c in numeric_cols:
    med = float(df[c].median())
    impute_vals['numeric'][c] = med
    df[c] = df[c].fillna(med)

for c in categorical_cols:
    mode_val = str(df[c].mode()[0])
    impute_vals['categorical'][c] = mode_val
    df[c] = df[c].fillna(mode_val)

print(f"Remaining missing values across entire dataset: {df.isnull().sum().sum()}")

## 4. Feature Engineering (One-Hot Encoding & X/Y Split)

In [ ]:
y = (df['Loan_Approved'] == 'Yes').astype(int)
X = df.drop(columns=['Loan_Approved'])

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
feature_names = list(X_encoded.columns)
print(f"Total model input features: {len(feature_names)}")
print("Sample features:", feature_names[:10])

## 5. Stratified Train-Test Splitting & Feature Normalization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape:  {X_test_scaled.shape}")

## 6. Model Training: Logistic Regression (L2 Regularized)

In [ ]:
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print("Model training completed successfully!")

## 7. Model Evaluation: Banking Risk Metrics

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy:   {acc*100:.2f}%")
print(f"ROC-AUC:    {auc:.4f} (Target >= 0.82: {'PASSED' if auc >= 0.82 else 'FAILED'})")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)']))

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title("Confusion Matrix — Credit Risk Baseline")
plt.show()

# ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#0066FF', lw=2, label=f'Logistic Regression (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='#94A3B8', linestyle='--')
plt.title("ROC-AUC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

## 8. Feature Explainability (Coefficients & Odds Ratios)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight': model.coef_[0],
    'Odds_Ratio': np.exp(model.coef_[0])
}).sort_values('Weight', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=coef_df, x='Weight', y='Feature', palette='coolwarm')
plt.title("Logistic Regression Feature Importance (Model Weights)")
plt.axvline(0, color='black', lw=0.8)
plt.show()

## 9. Serialization of Model Artifacts

In [ ]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

joblib.dump(model, os.path.join(models_dir, "logistic_regression.joblib"))
joblib.dump(scaler, os.path.join(models_dir, "scaler.joblib"))
with open(os.path.join(models_dir, "feature_names.json"), 'w') as f:
    json.dump(feature_names, f, indent=2)

print("All artifacts successfully exported to:", models_dir)